# RAG (krok po kroku)

Ten notebook przeprowadza krok po kroku przez cały proces RAG:
1. ładowanie retrievera i rerankera
2. indeksowanie dokumentów
3. retrieval i reranking
4. tworzenie promptu do LLM
5. generowanie odpowiedzi z modelu `gemma3:4b` przez Ollama


In [1]:
import os
from typing import List, Dict

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util

# Modela dla retriever/reranker oraz model Ollama
RETRIEVER_MODEL = "PKOBP/polish-roberta-8k"
RERANKER_MODEL = "sdadas/polish-reranker-roberta-v2"
LLM_MODEL = "gemma3:4b"

# Dane wejściowe.

documents = [
    "Umowę B2B można wypowiedzieć z zachowaniem 30-dniowego okresu wypowiedzenia.",
    "Pracownik zatrudniony na umowę o pracę ma okres wypowiedzenia zależny od stażu pracy.",
    "W ramach zatrudnienia na umowę B2B pracownika obowiazuje klauzula poufności.",
    "B2B to forma współpracy między firmami, gdzie jedna strona świadczy usługi lub dostarcza produkty drugiej stronie.",
    "Kontrakt może zostać rozwiązany za porozumieniem stron w dowolnym momencie.",
    "Faktury powinny być wystawiane do 10. dnia miesiąca.",
    "W przypadku naruszenia poufności spółka może rozwiązać współpracę ze skutkiem natychmiastowym."
]

query = "Jaki jest okres wypowiedzenia umowy B2B?"


/Users/lukasz/Documents/projects/rags/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Urządzenie (GPU/CPU)

Sprawdzamy, czy dostępne jest MPS (Apple GPU), inaczej używamy CPU.

In [2]:
def get_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Using device: {DEVICE}")

Using device: mps


## 4. Ładowanie retrievera i rerankera

Tworzymy indeksujący retriever i reranker. Ollama jest zewnętrznym serwisem, więc nie ładujemy go do pamięci.

In [3]:
retriever = SentenceTransformer(RETRIEVER_MODEL, device=DEVICE)
reranker = CrossEncoder(RERANKER_MODEL)

# Ollama działa z poziomu zapytania do serwisu.
# Upewnij się, że `ollama` jest zainstalowana i `phi3:mini` jest dostępny/ściągnięty.

No sentence-transformers model found with name PKOBP/polish-roberta-8k. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5925.65it/s]


## 5. Indeksowanie dokumentów

Tworzymy reprezentacje wektorowe dokumentów do wyszukiwania podobnych treści.

In [4]:
doc_embeddings = retriever.encode(
    documents,
    convert_to_tensor=True,
    normalize_embeddings=True
)


## 6. Retrieval

Znajdujemy najlepsze dokumenty na podstawie podobieństwa wektorowego query do fragmentów.

In [5]:
def retrieve(query: str, top_k: int = 5) -> List[Dict]:
    query_embedding = retriever.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    scores = util.cos_sim(query_embedding, doc_embeddings)[0]
    top_k = min(top_k, len(documents))
    top = torch.topk(scores, k=top_k)

    results = []
    for score, idx in zip(top.values, top.indices):
        results.append({
            "doc_id": int(idx),
            "text": documents[int(idx)],
            "retrieval_score": float(score)
        })
    return results


## 7. Reranking

Doprecyzowujemy listę kandydatów za pomocą cross encoder, aby uwzględnić relację pytanie–dokument.

In [6]:
def rerank(query: str, candidates: List[Dict], top_n: int = 3) -> List[Dict]:
    pairs = [[query, c["text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)

    for c, score in zip(candidates, rerank_scores):
        c["rerank_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]


## 8. Budowa promptu dla LLM

Sklejamy kontekst i pytanie w postać, którą przekażemy do modelu Ollama.

In [7]:
def build_messages(query: str, context_chunks: List[str]) -> List[Dict[str, str]]:
    context = "\n\n".join(
        [f"[Fragment {i+1}]\n{chunk}" for i, chunk in enumerate(context_chunks)]
    )

    system_message = (
        "Jesteś asystentem QA działającym w systemie RAG. "
        "Odpowiadaj wyłącznie po polsku i wyłącznie na podstawie dostarczonego kontekstu. "
        "Jeżeli w kontekście nie ma wystarczających danych, odpowiedz dokładnie: "
        "'Nie mam wystarczających danych w dostarczonym kontekście.' "
    )

    user_message = (
        f"Pytanie:\n{query}\n\n"
        f"Kontekst:\n{context}\n\n"
        "Podaj odpowiedź na podstawie fragmentów w dokumentach z kontekstu. "
        "Odpowiedź powinna być krótka i konkretna. "
        "Podaj identyfikator dokumentu jako: doc_id: FRAGMENT ID."
    )

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]


## 9. Generacja finalnej odpowiedzi

Wywołujemy model `gemma3:4b` przez Ollama (Python API albo CLI jako fallback).

In [8]:
def generate_answer(messages: List[Dict[str, str]], max_new_tokens: int = 1024) -> str:
    system_message = messages[0]["content"] if messages else ""
    user_message = messages[1]["content"] if len(messages) > 1 else ""
    prompt = f"{system_message}\n\n{user_message}"

    generated = ""

    try:
        print("Using Ollama client library...")
        from ollama import Client, Options
        client = Client()
        response = client.generate(
            model=LLM_MODEL,
            prompt=prompt,
            options=Options(num_predict=max_new_tokens, temperature=0.7),
            stream=False,
        )
        generated = response["response"].strip() if hasattr(response, "response") else str(response).strip()
    except Exception:
        print("Ollama client library failed, falling back to subprocess...")
        import subprocess
        result = subprocess.run(
            [
                "ollama", "generate", LLM_MODEL,
                "--temperature", "0.7",
                "--max-tokens", str(max_new_tokens),
                "--no-stream"
            ],
            input=prompt,
            capture_output=True,
            text=True,
            check=True,
        )
        generated = result.stdout.strip()

    generated = generated.split("\n")[0].strip()
    if generated.lower().startswith("odpowiedź:"):
        generated = generated[len("odpowiedź:"):].strip()

    return generated


## 10. Cały pipeline

Wykonujemy retrieval, reranking, budowę promptu i generację odpowiedzi.

In [9]:
retrieved = retrieve(query, top_k=5)

print("\n=== Po retrievalu ===")
for r in retrieved:
    print(f"{r['retrieval_score']:.4f} | {r['text']}")

reranked = rerank(query, retrieved, top_n=3)

print("\n=== Po rerankingu ===")
for r in reranked:
    print(
        f"rerank={r['rerank_score']:.4f} | "
        f"retrieve={r['retrieval_score']:.4f} | "
        f"{r['text']}"
    )

context_chunks = [r['text'] for r in reranked]
messages = build_messages(query, context_chunks)

print("\n=== Wiadomości do LLM ===")
for m in messages:
    print(f"\n[{m['role'].upper()}]\n{m['content']}")

answer = generate_answer(messages)

print("\n=== Finalna odpowiedź ===")
print(answer)



=== Po retrievalu ===
0.9595 | Umowę B2B można wypowiedzieć z zachowaniem 30-dniowego okresu wypowiedzenia.
0.9143 | Pracownik zatrudniony na umowę o pracę ma okres wypowiedzenia zależny od stażu pracy.
0.8990 | W ramach zatrudnienia na umowę B2B pracownika obowiazuje klauzula poufności.
0.8817 | B2B to forma współpracy między firmami, gdzie jedna strona świadczy usługi lub dostarcza produkty drugiej stronie.
0.8731 | Kontrakt może zostać rozwiązany za porozumieniem stron w dowolnym momencie.

=== Po rerankingu ===
rerank=1.0000 | retrieve=0.9595 | Umowę B2B można wypowiedzieć z zachowaniem 30-dniowego okresu wypowiedzenia.
rerank=0.9180 | retrieve=0.9143 | Pracownik zatrudniony na umowę o pracę ma okres wypowiedzenia zależny od stażu pracy.
rerank=0.8906 | retrieve=0.8731 | Kontrakt może zostać rozwiązany za porozumieniem stron w dowolnym momencie.

=== Wiadomości do LLM ===

[SYSTEM]
Jesteś asystentem QA działającym w systemie RAG. Odpowiadaj wyłącznie po polsku i wyłącznie na podst

In [10]:
query = "Jaka jest wartość przyspieszenia ziemskiego?"

retrieved = retrieve(query, top_k=5)

print("\n=== Po retrievalu ===")
for r in retrieved:
    print(f"{r['retrieval_score']:.4f} | {r['text']}")

reranked = rerank(query, retrieved, top_n=3)

print("\n=== Po rerankingu ===")
for r in reranked:
    print(
        f"rerank={r['rerank_score']:.4f} | "
        f"retrieve={r['retrieval_score']:.4f} | "
        f"{r['text']}"
    )

context_chunks = [r['text'] for r in reranked]
messages = build_messages(query, context_chunks)

print("\n=== Wiadomości do LLM ===")
for m in messages:
    print(f"\n[{m['role'].upper()}]\n{m['content']}")

answer = generate_answer(messages)

print("\n=== Finalna odpowiedź ===")
print(answer)



=== Po retrievalu ===
0.8242 | W przypadku naruszenia poufności spółka może rozwiązać współpracę ze skutkiem natychmiastowym.
0.8216 | B2B to forma współpracy między firmami, gdzie jedna strona świadczy usługi lub dostarcza produkty drugiej stronie.
0.8104 | Kontrakt może zostać rozwiązany za porozumieniem stron w dowolnym momencie.
0.8000 | Pracownik zatrudniony na umowę o pracę ma okres wypowiedzenia zależny od stażu pracy.
0.7986 | W ramach zatrudnienia na umowę B2B pracownika obowiazuje klauzula poufności.

=== Po rerankingu ===
rerank=0.0212 | retrieve=0.8216 | B2B to forma współpracy między firmami, gdzie jedna strona świadczy usługi lub dostarcza produkty drugiej stronie.
rerank=0.0014 | retrieve=0.8104 | Kontrakt może zostać rozwiązany za porozumieniem stron w dowolnym momencie.
rerank=0.0013 | retrieve=0.8000 | Pracownik zatrudniony na umowę o pracę ma okres wypowiedzenia zależny od stażu pracy.

=== Wiadomości do LLM ===

[SYSTEM]
Jesteś asystentem QA działającym w systemie 